# Denoising Comparison: Which Cleaning Helps the GDBN?

Puts the four denoisers side by side (run notebooks 14–17 first; this notebook loads their saved runs).

| Notebook | Denoiser | What it cleans | Linear? |
|---|---|---|---|
| 14 | **PCA** | removes the 5 biggest shared return patterns (fixed loadings) | yes |
| 15 | **IPCA** | removes 5 shared patterns, exposures from characteristics | yes |
| 16 | **Conditional autoencoder** (Gu–Kelly–Xiu) | removes 5 shared patterns, exposures from a neural network | no |
| 17 | **Denoising autoencoder** | rebuilds each stock's 25 features through a waist of 8 | no |

**Three questions:**
1. **How much noise did each remove?** (out-of-sample R²)
2. **Did the GDBN predict better?** (rank IC, gross Sharpe, breakeven, especially 2021–2022)
3. **Did the connections get cleaner?** (fewer arrows, more stable from quarter to quarter)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pit_common as pc
import denoise_common as dc

D = pc.load()
METHODS = {"pca": "PCA", "ipca": "IPCA", "cae": "CAE", "dae": "DAE"}
runs = {}
for key, name in [("raw_daily", "raw daily"), ("raw_slow", "raw slow")] + \
                 [(f"{k}_{kind}", f"{v} {kind}") for k, v in METHODS.items() for kind in ["daily", "slow"]]:
    path = dc.OUT / f"wf_{key}.pkl"
    if path.exists():
        runs[name] = pickle.load(open(path, "rb"))
    else:
        print(f"missing {path.name}: run its notebook first")
print("Loaded:", list(runs))

## 1. How much did each denoiser remove? (out-of-sample R², average over 2006–2022)

In [ ]:
r2_rows = {}
for key, name in METHODS.items():
    path = dc.OUT / f"r2_{key}.csv"
    if path.exists():
        t = pd.read_csv(path, index_col="year")
        r2_rows[name] = t[[c for c in t.columns if c.startswith("R2")]].mean()
pd.DataFrame(r2_rows).T.round(3)

For PCA / IPCA / CAE, R² is the share of daily excess-return moves removed as "shared" (higher = more removed). For the DAE, it is how much of each feature the cleaned version keeps.

## 2. GDBN results

In [ ]:
table, scores = dc.evaluate(D, runs)
order = [n for kind in ["daily", "slow"] for n in table.index if n.endswith(kind)]
table = table.loc[order]
print(table.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, kind in zip(axes[:2], ["daily", "slow"]):
    for name, s in scores.items():
        if name.endswith(kind):
            ic = pc.rank_ic(s, pc.forward_excess(D, s, 1))
            ax.plot(ic.cumsum(), label=name, ls="--" if name.startswith("raw") else "-", lw=2 if name.startswith("raw") else 1.2)
    ax.axvline(pd.Timestamp("2021-01-01"), color="gray", lw=1)
    ax.set_title(f"Cumulative rank IC: {kind} inputs"); ax.legend(fontsize=8)
x = np.arange(len(table))
axes[2].bar(x - 0.2, table["arrow stability"], 0.4, label="at its own λ", color="lightgray")
axes[2].bar(x + 0.2, table["stability @ raw λ"], 0.4, label="at the raw GDBN's λ (fair)", color="C0")
for i, a in enumerate(table["arrows/quarter"]):
    axes[2].text(i - 0.2, table["arrow stability"].iloc[i], f"{a:.0f}", ha="center", va="bottom", fontsize=6)
axes[2].set_xticks(x); axes[2].set_xticklabels(table.index, rotation=45, ha="right", fontsize=8)
axes[2].legend(fontsize=7); axes[2].set_title("Arrow stability (numbers = arrows per quarter at its own λ)")
plt.tight_layout(); plt.show()

**Reading it:**
- **Prediction:** a denoiser helps if its rows beat the raw row **of the same kind** (daily vs daily, slow vs slow), above all in **2021–2022** (the only clean check).
- **Connections:** compare **stability @ raw λ** (blue bars): at the same price per arrow, does cleaning make the GDBN find more **persistent** links? The gray bars are inflated for versions that chose bigger networks.
- **Linear vs nonlinear:** if PCA/IPCA do as well as the CAE/DAE, the simpler, interpretable denoiser is the better choice for the thesis.

In [ ]:
table.to_csv(dc.OUT / "comparison.csv")
print("Saved to", dc.OUT / "comparison.csv")